# Notebook 02: Data Quality Audit & Hygiene Diagnostic
**Focus:** Integrity auditing, duplicate detection, orphan foreign keys, impossible quantities, and join explosions.


In [ ]:
import duckdb
import pandas as pd
con = duckdb.connect()
raw_dir = '../data/raw'


### 1. Duplicate Row Checks Across Tables


In [ ]:
# Checking coupon.csv exact duplicates
total_coups = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_dir}/coupon.csv')").fetchone()[0]
distinct_coups = con.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT * FROM read_csv_auto('{raw_dir}/coupon.csv'))").fetchone()[0]
print(f'coupon.csv: Total={total_coups:,}, Distinct={distinct_coups:,}, Duplicates={total_coups - distinct_coups:,}')


### 2. Causal Data Multi-Display Cartesian Risk


In [ ]:
causal_dups = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT PRODUCT_ID, STORE_ID, WEEK_NO 
        FROM read_csv_auto('{raw_dir}/causal_data.csv') 
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO 
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f'Multi-location display conflict groups in causal_data: {causal_dups:,}')


### 3. Numerical Value Ranges & Outliers in Transactions


In [ ]:
con.execute(f"""
    SELECT 
        MIN(SALES_VALUE) as min_sales, MAX(SALES_VALUE) as max_sales,
        MIN(QUANTITY) as min_qty, MAX(QUANTITY) as max_qty,
        COUNT(CASE WHEN SALES_VALUE = 0 THEN 1 END) as zero_sales,
        COUNT(CASE WHEN QUANTITY = 0 THEN 1 END) as zero_qty
    FROM read_csv_auto('{raw_dir}/transaction_data.csv')
""").df()
